# Topic 2: Writing Data & File Formats

> **Phase 5 → Week 8 → Topic 2**

---

## The DataFrameWriter API

```
df.write
  .format("parquet" | "delta" | "csv" | "json" | "orc" | "jdbc")
  .mode("overwrite" | "append" | "ignore" | "error")
  .option(key, value)
  .partitionBy("col1", "col2")     ← directory partitioning
  .bucketBy(N, "col")              ← file bucketing (requires saveAsTable)
  .sortBy("col")                   ← sort within bucket
  .saveAsTable("table_name")       ← write to Hive/catalog table
  .save("/path")                   ← write to path
```

---

## Write Modes

```
overwrite  → delete existing data, write fresh  (destructive!)
append     → add new files alongside existing   (non-destructive)
ignore     → no-op if path already exists       (silent skip)
error      → throw exception if path exists     (default, safe)
```

---

## Interview Cheat Sheet

**Q: How many output files does Spark create and how do you control it?**
> Spark creates one file per partition. If you have 200 shuffle partitions, you get 200 output files. To control it: (1) `coalesce(N)` before write — reduces partition count without a shuffle (only decreases). (2) `repartition(N)` before write — full shuffle to exactly N partitions. (3) `spark.sql.adaptive.enabled=true` with AQE coalescing auto-tunes partition count. Rule: aim for 128-256MB per output file.

**Q: What is the difference between `partitionBy()` and `bucketBy()` when writing?**
> `partitionBy("col")` creates subdirectories per distinct value of the column (e.g., `status=delivered/`, `status=pending/`). Good for partition pruning on queries. Creates one directory per value — bad for high-cardinality columns (10k unique customer_ids = 10k directories). `bucketBy(N, "col")` creates N fixed-size files, each containing rows hashing to that bucket. Good for joins without shuffle — future joins on the same key skip the Exchange. Requires `saveAsTable()`, not `save()`.

**Q: Which file format should you use in production and why?**
> Parquet for read-heavy analytics (columnar, compressed, fast for OLAP queries). Delta Lake for tables that are updated/deleted (ACID, time travel, MERGE). ORC is similar to Parquet but preferred in Hive ecosystems. Avoid CSV/JSON in production for large data — not splittable efficiently, no schema enforcement, no compression metadata.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os
import subprocess

spark = SparkSession.builder \
    .appName("Week8 - Writing Formats") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

orders = spark.read.csv("/workspace/week4/data/orders.csv", header=True, inferSchema=True)
print(f"Loaded {orders.count()} orders, {orders.rdd.getNumPartitions()} partitions")

## Part 1: Write Modes

In [ ]:
# Demonstrate all 4 write modes

# overwrite: replaces existing data completely
orders.write.mode("overwrite").parquet("/tmp/mode_demo")
print("overwrite: wrote fresh data")

# append: adds new files alongside existing (does NOT check for duplicates!)
orders.write.mode("append").parquet("/tmp/mode_demo")
df_appended = spark.read.parquet("/tmp/mode_demo")
print(f"append: now has {df_appended.count()} rows (doubled — no dedup!)")

# ignore: no-op if path already exists
extra = spark.createDataFrame([("X999",)], ["order_id"])
extra.write.mode("ignore").parquet("/tmp/mode_demo")
df_after_ignore = spark.read.parquet("/tmp/mode_demo")
print(f"ignore: still {df_after_ignore.count()} rows (extra data was ignored)")

# error (default): throws if path exists
try:
    orders.write.parquet("/tmp/mode_demo")  # no mode = error
except Exception as e:
    print(f"error mode (default): {type(e).__name__} raised — path already exists")

## Part 2: Controlling Output File Count

In [ ]:
def count_files(path):
    result = subprocess.run(["find", path, "-name", "*.parquet"], 
                            capture_output=True, text=True)
    files = [f for f in result.stdout.strip().split("\n") if f]
    return len(files)

# Default: one file per partition
large_df = spark.range(100_000).withColumn("val", F.rand())
large_df.write.mode("overwrite").parquet("/tmp/file_count/default")
print(f"Default ({large_df.rdd.getNumPartitions()} partitions): {count_files('/tmp/file_count/default')} files")

# coalesce: reduce file count without shuffle
large_df.coalesce(2).write.mode("overwrite").parquet("/tmp/file_count/coalesce")
print(f"coalesce(2): {count_files('/tmp/file_count/coalesce')} files (no shuffle)")

# repartition: full shuffle to exact partition count
large_df.repartition(3).write.mode("overwrite").parquet("/tmp/file_count/repartition")
print(f"repartition(3): {count_files('/tmp/file_count/repartition')} files (full shuffle)")

print()
print("Rule: use coalesce() before write to reduce file count (no shuffle cost)")
print("      use repartition() only when you need balanced or re-keyed partitions")

## Part 3: Compression Options

In [ ]:
import time

df = spark.range(500_000) \
    .withColumn("name", F.concat(F.lit("user_"), (F.col("id") % 1000).cast("string"))) \
    .withColumn("city", F.element_at(F.array(F.lit("NYC"), F.lit("LA"), F.lit("Chicago")),
                                     (F.col("id") % 3 + 1).cast("int")))

for codec in ["snappy", "gzip", "none"]:
    path = f"/tmp/compression/{codec}"
    t0 = time.time()
    df.write.mode("overwrite") \
      .option("compression", codec) \
      .parquet(path)
    elapsed = time.time() - t0
    result = subprocess.run(["du", "-sh", path], capture_output=True, text=True)
    size = result.stdout.split()[0]
    print(f"  {codec:10s}: {size:8s}  write time: {elapsed:.2f}s")

print()
print("""
Compression tradeoffs:
  snappy  → fast read/write, moderate compression. DEFAULT for most cases.
  gzip    → smaller files, slower CPU. Use for cold storage / archival.
  lz4     → fastest CPU, least compression. Use for hot data / streaming.
  zstd    → best balance (Spark 3.0+). Good general-purpose replacement for snappy.
  none    → no compression. Only if downstream system doesn't support codecs.
""")

## Part 4: partitionBy — Directory Partitioning

In [ ]:
# partitionBy creates directory hierarchy: /path/col=value/
orders.write.mode("overwrite") \
      .partitionBy("status") \
      .parquet("/tmp/orders_partitioned_by_status")

result = subprocess.run(["find", "/tmp/orders_partitioned_by_status", "-type", "d"],
                        capture_output=True, text=True)
print("Directory structure:")
for line in sorted(result.stdout.strip().split("\n")):
    print(f"  {line}")

# Read with partition pruning
df_partitioned = spark.read.parquet("/tmp/orders_partitioned_by_status")
print(f"\nAll records: {df_partitioned.count()}")

# Filter uses partition pruning — only reads status=delivered directory
delivered = df_partitioned.filter(F.col("status") == "delivered")
print(f"Delivered only: {delivered.count()} (only that directory was read)")

In [ ]:
# HIGH CARDINALITY WARNING: partitionBy on customer_id
# This is a BAD idea for high-cardinality columns!
print("""
partitionBy Anti-Pattern — High Cardinality:
═══════════════════════════════════════════════════════
BAD: partitionBy('customer_id') if you have 1M customers
  → 1M directories × N files each = millions of tiny files
  → HDFS NameNode metadata overflow
  → Spark driver runs out of memory listing files
  → Each task has almost no data (scheduling overhead > compute)

RULE: Only partitionBy columns with LOW cardinality:
  ✓ date/year/month/day (daily ETL → ~365 partitions)
  ✓ status (delivered/pending/cancelled → 3-10 partitions)
  ✓ region/country (reasonable bounded set)
  ✗ customer_id, user_id, transaction_id (unbounded high cardinality)

For high-cardinality join optimization: use bucketBy() instead.
═══════════════════════════════════════════════════════
""")

## Part 5: Format Comparison

In [ ]:
print("""
File Format Decision Guide:
════════════════════════════════════════════════════════════════
Format   Splittable  Schema   Compress  Updates  Use Case
───────  ──────────  ───────  ────────  ───────  ──────────────────────────────────
Parquet  ✓           Built-in  ✓         ✗        Analytics, data lake, read-heavy
Delta    ✓           Built-in  ✓         ✓        Mutable tables, CDC, upserts
ORC      ✓           Built-in  ✓         ✗        Hive-optimized (prefer Parquet)
Avro     ✓           Built-in  ✓         ✗        Kafka/streaming, row-oriented
JSON     ✓ (GZIP=✗)  ✗         opt-in    ✗        APIs, semi-structured (avoid prod)
CSV      ✓ (GZIP=✗)  ✗         opt-in    ✗        Exchange with other systems only

Production Default:
  ✅ Parquet + Snappy  → standard analytics (90% of use cases)
  ✅ Delta Lake        → any table that needs UPDATE/DELETE/MERGE
  ✅ Avro             → Kafka source/sink
  ✅ Parquet + ZSTD   → cold storage / archival

Never in production (for large datasets):
  ❌ CSV             → no schema, slow to parse, can't do column pruning
  ❌ JSON            → verbose, no schema, can't do column pushdown
════════════════════════════════════════════════════════════════
""")

## Exercises

1. Write the orders DataFrame to Parquet with `snappy` compression. Then write it again with `gzip`. Compare file sizes.
2. Write orders partitioned by `status` to Parquet. Read it back. Using `explain()`, confirm partition pruning activates when filtering on `status`.
3. What is the problem with this code? `orders.repartition(1000).write.mode('overwrite').parquet('/output')`
4. Write orders to CSV and to Parquet. Compare the file sizes. Why is Parquet smaller?
5. When would you use `write.mode('append')` vs `write.mode('overwrite')`? Give a real-world ETL scenario for each.

In [ ]:
# Exercise 4: CSV vs Parquet size comparison
orders.write.mode("overwrite").option("header", "true").csv("/tmp/orders_csv_format")
orders.write.mode("overwrite").parquet("/tmp/orders_parquet_format")

for path, fmt in [("/tmp/orders_csv_format", "CSV"), ("/tmp/orders_parquet_format", "Parquet")]:
    result = subprocess.run(["du", "-sh", path], capture_output=True, text=True)
    size = result.stdout.split()[0]
    print(f"  {fmt:10s}: {size}")

print("""
Parquet is smaller because:
  1. Columnar: stores each column together → similar values compress better
  2. Dictionary encoding: repeated values (status, category) stored as integers
  3. Run-length encoding: long runs of same value stored as count×value
  4. Snappy compression applied on top
CSV stores each row as text with delimiters — no encoding, no compression.
""")